# Geometry foundations lab

A hands-on companion to the [geometry foundations learning path](../learning-paths/geometry-foundations.md).

Move a point, change a segment and shift a polygon. The controls let you see how the answer changes as the geometry changes.

This lab brings three related questions together:
- Is a point inside, outside or on a polygon boundary?
- Do two segments miss, cross, touch or overlap?
- What geometry do two areas share?

The examples use small planar coordinates so the geometry is easy to inspect. They are not georeferenced map data.

Open this notebook in [Google Colab](https://colab.research.google.com/github/geo-dev-hub/spatial-analysis/blob/main/notebooks/geometry-foundations-lab.ipynb) or run it in JupyterLab with the repository notebook requirements.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from ipywidgets import IntSlider, interact
from shapely import box

current = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (current, *current.parents)
        if (candidate / "implementations/python/point-in-polygon").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Run this notebook from a checkout of geo-dev-hub/spatial-analysis."
    )

for relative_path in (
    "implementations/python/point-in-polygon",
    "implementations/python/segment-intersection",
    "implementations/python/polygon-intersection",
):
    sys.path.insert(0, str((repo_root / relative_path).resolve()))

from point_in_polygon import locate_point
from segment_intersection import segment_intersection
from intersection_demo import polygon_intersection

print("Repository:", repo_root)

## 1. Move a point around a polygon

The polygon below is concave: it has an inward corner. Move the query point and watch the classification change between **inside**, **outside** and **boundary**.

The ray-crossing implementation is deliberately small. It handles one simple ring and uses exact floating-point comparisons; it is a teaching example, not a robust replacement for a geometry engine.

In [ ]:
concave_ring = [(0, 0), (4, 0), (4, 4), (2, 2), (0, 4)]

def explore_point_location(x=1, y=2):
    point = (x, y)
    location = locate_point(point, concave_ring)
    closed_ring = concave_ring + [concave_ring[0]]
    xs, ys = zip(*closed_ring)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.fill(xs, ys, alpha=0.15, label="Polygon")
    ax.plot(xs, ys, linewidth=2)
    ax.scatter([x], [y], s=100, label="Query point")
    ax.set_xlim(-1, 5)
    ax.set_ylim(-1, 5)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Point ({x}, {y}): {location.value}")
    ax.grid(True)
    ax.legend()
    plt.show()
    print(f"Classification: {location.value}")

interact(
    explore_point_location,
    x=IntSlider(value=1, min=-1, max=5, step=1, description="Point x"),
    y=IntSlider(value=2, min=-1, max=5, step=1, description="Point y"),
);

## 2. Change the second segment

The first segment stays fixed from A to B. Move C and D to test a second segment.

The result distinguishes no intersection, a single point and a collinear overlap. The implementation accepts integer endpoints and constructs crossing coordinates exactly with rational arithmetic.

In [ ]:
segment_a = (0, 0)
segment_b = (4, 4)

def explore_segment_intersection(cx=0, cy=4, dx=4, dy=0):
    segment_c = (cx, cy)
    segment_d = (dx, dy)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(
        [segment_a[0], segment_b[0]],
        [segment_a[1], segment_b[1]],
        linewidth=3,
        label="Segment AB",
    )
    ax.plot(
        [segment_c[0], segment_d[0]],
        [segment_c[1], segment_d[1]],
        linewidth=3,
        label="Segment CD",
    )
    ax.scatter(
        [segment_a[0], segment_b[0], segment_c[0], segment_d[0]],
        [segment_a[1], segment_b[1], segment_c[1], segment_d[1]],
        s=35,
    )

    try:
        result = segment_intersection(segment_a, segment_b, segment_c, segment_d)
    except ValueError as error:
        result = None
        print(f"Invalid input: {error}")

    if result is not None and result.kind == "point":
        px, py = result.geometry
        ax.scatter([float(px)], [float(py)], marker="x", s=130, linewidth=3, label="Intersection")
    elif result is not None and result.kind == "overlap":
        start, end = result.geometry
        ax.plot(
            [start[0], end[0]],
            [start[1], end[1]],
            linewidth=7,
            label="Shared segment",
        )

    ax.set_xlim(-1, 5)
    ax.set_ylim(-1, 5)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Segment intersection")
    ax.grid(True)
    ax.legend()
    plt.show()

    if result is not None:
        print("Result:", result.kind)
        print("Exact geometry:", result.geometry)

interact(
    explore_segment_intersection,
    cx=IntSlider(value=0, min=-1, max=5, step=1, description="C x"),
    cy=IntSlider(value=4, min=-1, max=5, step=1, description="C y"),
    dx=IntSlider(value=4, min=-1, max=5, step=1, description="D x"),
    dy=IntSlider(value=0, min=-1, max=5, step=1, description="D y"),
);

## 3. Change the overlap between two areas

The first rectangle stays fixed. Move the second rectangle and change its width.

The output may be a polygon, line, point or empty geometry. This example calls the Shapely/GEOS overlay operation; it does not reimplement a complete polygon-overlay engine.

In [ ]:
def draw_geometry(ax, geometry, color, fill=False):
    if geometry.is_empty:
        return

    if geometry.geom_type == "Polygon":
        coordinates = list(geometry.exterior.coords)
        patch = MplPolygon(
            coordinates,
            closed=True,
            edgecolor=color,
            facecolor=color if fill else "none",
            alpha=0.25 if fill else 1.0,
            linewidth=2,
        )
        ax.add_patch(patch)
        for interior in geometry.interiors:
            hole_x, hole_y = zip(*interior.coords)
            ax.plot(hole_x, hole_y, color=color, linewidth=1.5)
    elif geometry.geom_type in ("LineString", "LinearRing"):
        line_x, line_y = geometry.xy
        ax.plot(line_x, line_y, color=color, linewidth=4)
    elif geometry.geom_type == "Point":
        ax.scatter([geometry.x], [geometry.y], color=color, s=90)
    elif hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            draw_geometry(ax, part, color, fill=fill)

def explore_polygon_intersection(offset_x=2, offset_y=1, width=3):
    first = box(0, 0, 4, 4)
    second = box(offset_x, offset_y, offset_x + width, offset_y + 2)
    shared = polygon_intersection(first, second)

    fig, ax = plt.subplots(figsize=(6, 5))
    draw_geometry(ax, first, "C0")
    draw_geometry(ax, second, "C1")
    draw_geometry(ax, shared, "C2", fill=True)
    ax.set_xlim(-2, 11)
    ax.set_ylim(-2, 7)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Intersection: {shared.geom_type} · area {shared.area:g}")
    ax.grid(True)
    plt.show()

    print("Result type:", shared.geom_type)
    print("Area:", shared.area)
    print("WKT:", shared.wkt)

interact(
    explore_polygon_intersection,
    offset_x=IntSlider(value=2, min=-1, max=5, step=1, description="Second x"),
    offset_y=IntSlider(value=1, min=-1, max=5, step=1, description="Second y"),
    width=IntSlider(value=3, min=1, max=5, step=1, description="Width"),
);

## What to notice

- A point on an edge is a separate result, not an ordinary inside point.
- A ray that passes through a polygon vertex needs a consistent crossing rule.
- Segment contact includes more than proper crossings: endpoints and collinear overlaps matter.
- Polygon intersection is a set operation. Its output dimension depends on how the inputs meet.
- The pairwise segment routine answers one pair. The [Bentley–Ottmann lesson](../algorithms/geometry/bentley-ottmann.md) explains how to find intersections across a whole set.

## Go further

- [Point in polygon: explanation and implementation](../algorithms/geometry/point-in-polygon.md)
- [Segment intersection: explanation and implementation](../algorithms/geometry/segment-intersection.md)
- [Polygon intersection: explanation and implementation](../algorithms/geometry/polygon-intersection.md)
- [Geometry foundations learning path](../learning-paths/geometry-foundations.md)
- [Python implementation walkthroughs](../implementations/README.md)

Change one thing at a time, predict what will happen, then move the controls and check your reasoning.